# Clase 27: Detección de Anomalías

**Idea central:** La detección de anomalías encuentra los puntos que no siguen el patrón esperado — clave para detectar fraudes, fallos o eventos raros.

En esta clase aprenderemos a:
- Distinguir anomalías de outliers estadísticos
- Aplicar métodos estadísticos: IQR y Z-score
- Usar Isolation Forest y Local Outlier Factor de sklearn
- Visualizar y comparar los resultados de cada método

In [ ]:
# Celda 1: Imports y configuración
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

print('Librerías cargadas correctamente')

In [ ]:
# Celda 2: Cargar y explorar el dataset de ventas
df_ventas = pd.read_csv('../../datasets/ventas_tienda.csv')

print('Forma del dataset de ventas:', df_ventas.shape)
print('\nColumnas:', df_ventas.columns.tolist())
print('\nEstadísticas descriptivas de total_neto:')
print(df_ventas['total_neto'].describe())

# Visualización inicial
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_ventas['total_neto'].plot(ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Ventas diarias (total_neto)')
axes[0].set_xlabel('Índice')
axes[0].set_ylabel('Total neto ($)')
axes[0].grid(True, alpha=0.3)

df_ventas['total_neto'].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_title('Distribución de ventas (boxplot)')
axes[1].set_ylabel('Total neto ($)')

plt.tight_layout()
plt.show()
print('Los puntos fuera de los bigotes del boxplot son candidatos a outliers.')

In [ ]:
# Celda 3: Método 1 — Regla del IQR
Q1 = df_ventas['total_neto'].quantile(0.25)
Q3 = df_ventas['total_neto'].quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print('=== Método IQR ===')
print(f'Q1: {Q1:.2f}')
print(f'Q3: {Q3:.2f}')
print(f'IQR: {IQR:.2f}')
print(f'Límite inferior: {limite_inf:.2f}')
print(f'Límite superior: {limite_sup:.2f}')

df_ventas['outlier_iqr'] = (
    (df_ventas['total_neto'] < limite_inf) |
    (df_ventas['total_neto'] > limite_sup)
)

n_iqr = df_ventas['outlier_iqr'].sum()
print(f'\nAnomalías detectadas: {n_iqr} ({n_iqr/len(df_ventas):.1%} del total)')
print('\nDías anómalos:')
print(df_ventas[df_ventas['outlier_iqr']][['total_neto']].to_string())

In [ ]:
# Celda 4: Método 2 — Z-score
z_scores = np.abs(stats.zscore(df_ventas['total_neto']))
df_ventas['z_score'] = z_scores
df_ventas['outlier_zscore'] = z_scores > 3

n_zscore = df_ventas['outlier_zscore'].sum()
print('=== Método Z-score (umbral: |z| > 3) ===')
print(f'Anomalías detectadas: {n_zscore} ({n_zscore/len(df_ventas):.1%} del total)')
print('\nTop 5 valores con mayor Z-score:')
print(df_ventas.nlargest(5, 'z_score')[['total_neto', 'z_score']].to_string())

# Comparación IQR vs Z-score
print('\n--- Comparación IQR vs Z-score ---')
ambos = (df_ventas['outlier_iqr'] & df_ventas['outlier_zscore']).sum()
solo_iqr = (df_ventas['outlier_iqr'] & ~df_ventas['outlier_zscore']).sum()
solo_zscore = (~df_ventas['outlier_iqr'] & df_ventas['outlier_zscore']).sum()
print(f'Detectados por ambos métodos: {ambos}')
print(f'Solo por IQR:                 {solo_iqr}')
print(f'Solo por Z-score:             {solo_zscore}')

In [ ]:
# Celda 5: Método 3 — Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # esperamos ~5% de anomalías
    random_state=42
)

df_ventas['anomalia_iso'] = iso_forest.fit_predict(df_ventas[['total_neto']])
# fit_predict devuelve: 1 = normal, -1 = anomalía

n_iso = (df_ventas['anomalia_iso'] == -1).sum()
print('=== Isolation Forest ===')
print(f'Anomalías detectadas: {n_iso} ({n_iso/len(df_ventas):.1%} del total)')

# Visualización
fig, ax = plt.subplots(figsize=(14, 5))

normales = df_ventas[df_ventas['anomalia_iso'] == 1]
anomalas = df_ventas[df_ventas['anomalia_iso'] == -1]

ax.scatter(normales.index, normales['total_neto'],
           c='steelblue', s=20, alpha=0.6, label='Normal')
ax.scatter(anomalas.index, anomalas['total_neto'],
           c='red', s=80, zorder=5, label=f'Anomalía ({n_iso})')
ax.axhline(limite_sup, color='orange', linestyle='--',
           alpha=0.7, label=f'Límite IQR superior ({limite_sup:.0f})')
ax.axhline(limite_inf, color='orange', linestyle='--', alpha=0.7)
ax.set_title('Ventas diarias — Anomalías detectadas por Isolation Forest')
ax.set_xlabel('Día')
ax.set_ylabel('Total neto ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 6: Método 4 — Local Outlier Factor (LOF) en datos de transporte
df_transporte = pd.read_csv('../../datasets/transporte.csv')

print('Dataset de transporte:')
print(df_transporte.shape)
print(df_transporte['retraso_min'].describe())

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

df_transporte['anomalia_lof'] = lof.fit_predict(df_transporte[['retraso_min']])
df_transporte['lof_score'] = lof.negative_outlier_factor_

n_lof = (df_transporte['anomalia_lof'] == -1).sum()
print(f'\nAnomalías LOF detectadas: {n_lof}')
print('\nTop 5 retrasos más anómalos:')
print(df_transporte.nsmallest(5, 'lof_score')[['retraso_min', 'lof_score', 'anomalia_lof']].to_string())

In [ ]:
# Celda 7: Visualización de anomalías en retrasos de transporte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de retrasos
df_transporte['retraso_min'].plot(kind='box', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de retrasos (minutos)')
axes[0].set_ylabel('Retraso (min)')

# Serie temporal con anomalías
norm_t = df_transporte[df_transporte['anomalia_lof'] == 1]
anom_t = df_transporte[df_transporte['anomalia_lof'] == -1]

axes[1].scatter(norm_t.index, norm_t['retraso_min'],
                c='steelblue', s=20, alpha=0.6, label='Normal')
axes[1].scatter(anom_t.index, anom_t['retraso_min'],
                c='red', s=80, zorder=5, label=f'Anomalía LOF ({n_lof})')
axes[1].set_title('Retrasos de transporte — Anomalías LOF')
axes[1].set_xlabel('Índice')
axes[1].set_ylabel('Retraso (min)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 8: Comparación de todos los métodos en ventas
comparacion = pd.DataFrame({
    'total_neto': df_ventas['total_neto'],
    'IQR': df_ventas['outlier_iqr'].astype(int),
    'Z-score': df_ventas['outlier_zscore'].astype(int),
    'Isolation Forest': (df_ventas['anomalia_iso'] == -1).astype(int)
})

comparacion['votos'] = comparacion[['IQR', 'Z-score', 'Isolation Forest']].sum(axis=1)

print('Resumen de detecciones por método:')
print(comparacion[['IQR', 'Z-score', 'Isolation Forest']].sum().to_string())

print('\nPuntos detectados por los 3 métodos (más confiables):')
consenso = comparacion[comparacion['votos'] == 3]
print(f'Total: {len(consenso)} puntos')
print(consenso[['total_neto', 'IQR', 'Z-score', 'Isolation Forest']].to_string())

# Resumen visual
metodos = ['IQR', 'Z-score', 'Isolation Forest']
counts = [comparacion['IQR'].sum(), comparacion['Z-score'].sum(),
          comparacion['Isolation Forest'].sum()]

plt.figure(figsize=(7, 4))
plt.bar(metodos, counts, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
plt.title('Número de anomalías detectadas por método')
plt.ylabel('Cantidad de anomalías')
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(counts):
    plt.text(i, v + 0.1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Celda 9: Guía para elegir el método correcto
guia = pd.DataFrame({
    'Método': ['IQR', 'Z-score', 'Isolation Forest', 'Local Outlier Factor'],
    'Tipo': ['Estadístico', 'Estadístico', 'ML no supervisado', 'ML no supervisado'],
    'Supuesto': ['Ninguno', 'Distribución normal', 'Ninguno', 'Ninguno'],
    'Dimensiones': ['Una variable', 'Una variable', 'Múltiples', 'Múltiples'],
    'Velocidad': ['Muy rápido', 'Muy rápido', 'Rápido', 'Más lento'],
    'Interpretabilidad': ['Alta', 'Alta', 'Media', 'Media']
})

print(guia.to_string(index=False))
print('\nRecomendación:')
print('- Datos simples, necesitas explicar el resultado: usa IQR')
print('- Datos normales, una variable: usa Z-score')
print('- Datos complejos, muchas variables, dataset grande: usa Isolation Forest')
print('- Anomalías locales, densidad variable en el espacio: usa LOF')

## Resumen de la clase

| Método | Qué hace | Cuándo usarlo |
|---|---|---|
| **IQR** | Detecta puntos fuera de Q1-1.5×IQR y Q3+1.5×IQR | Simple, datos sesgados, una variable |
| **Z-score** | Detecta puntos a más de 3σ de la media | Datos normales, una variable |
| **Isolation Forest** | Aísla anomalías con cortes aleatorios en árboles | Muchas variables, dataset grande |
| **LOF** | Compara densidad local vs vecinos | Anomalías locales, densidad variable |

**Recuerda:** Siempre valida las anomalías detectadas con expertos del dominio antes de actuar. Lo estadísticamente inusual no siempre es un error — puede ser un evento legítimo y raro.